In [21]:
from qiskit import quantum_info as qi
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.colors as colors
from scipy.optimize import root

import sympy as sp
from utils import get_poly_from_code, poly_dicts_to_sympy

In [44]:
def find_magic_fixed_points(
    generator_set,
    logical_operator,
    normalize=False,
    n_grid=11,
    n_random=500,
    seed=0,
    residual_tol=1e-10,
    dedup_tol=1e-7,
    stability_tol=1e-8,
    ball_tol=1e-8,
    psuc_tol=1e-12,
):
    """
    Find and classify fixed points of a 3D Bloch map T(x,y,z).

    Parameters
    ----------
    generator_set : list
        Stabilizer generators, passed into your get_poly_from_code function.

    logical_operator : dict
        Logical operator choice, passed into your get_poly_from_code function.

    normalize : bool
        If False, assumes Tx_sp, Ty_sp, Tz_sp are already normalized Bloch maps.
        If True, replaces Tx, Ty, Tz by Tx/psuc, Ty/psuc, Tz/psuc.

    n_grid : int
        Number of grid points per coordinate direction for deterministic seeds.

    n_random : int
        Number of random seeds inside the Bloch ball.

    Returns
    -------
    fixed_points : list of dict
        Each dict contains point, stability data, magic-state classification, etc.
    """

    x, y, z = sp.symbols("x y z", real=True)

    poly_dicts = get_poly_from_code(
        generator_set=generator_set,
        logical_operator=logical_operator
    )

    psuc_sp, Tx_sp, Ty_sp, Tz_sp = poly_dicts_to_sympy(poly_dicts)

    print(f"Logical Operator: {logical_operator}")

    # If Tx, Ty, Tz are unnormalized numerators, divide by success probability.
    if normalize:
        Tx_sp = sp.simplify(Tx_sp / psuc_sp)
        Ty_sp = sp.simplify(Ty_sp / psuc_sp)
        Tz_sp = sp.simplify(Tz_sp / psuc_sp)

    T_sp = sp.Matrix([Tx_sp, Ty_sp, Tz_sp])

    F_sp = sp.Matrix([
        Tx_sp - x,
        Ty_sp - y,
        Tz_sp - z,
    ])

    J_sp = T_sp.jacobian([x, y, z])

    F_func = sp.lambdify((x, y, z), F_sp, modules="numpy")
    J_func = sp.lambdify((x, y, z), J_sp, modules="numpy")
    T_func = sp.lambdify((x, y, z), T_sp, modules="numpy")
    psuc_func = sp.lambdify((x, y, z), psuc_sp, modules="numpy")

    def to_real_array(value, shape=None):
        arr = np.asarray(value)
        arr = np.real_if_close(arr, tol=1000)
        arr = np.asarray(arr, dtype=float)

        if shape is not None and arr.shape == ():
            arr = np.full(shape, float(arr))

        return arr

    def F_np(r):
        try:
            psuc_val = psuc_np(r)

            if not np.isfinite(psuc_val):
                return np.array([1e9, 1e9, 1e9])

            if abs(psuc_val) < 1e-10:
                return np.array([1e9, 1e9, 1e9])

            with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
                val = F_func(r[0], r[1], r[2])

            val = to_real_array(val).reshape(3)

            if not np.all(np.isfinite(val)):
                return np.array([1e9, 1e9, 1e9])

            return val

        except Exception:
            return np.array([1e9, 1e9, 1e9])

    def J_np(r):
        val = J_func(r[0], r[1], r[2])
        return to_real_array(val).reshape(3, 3)

    def T_np(r):
        val = T_func(r[0], r[1], r[2])
        return to_real_array(val).reshape(3)

    def psuc_np(r):
        try:
            val = psuc_func(r[0], r[1], r[2])
            val = float(np.real_if_close(val))
            return val
        except Exception:
            return np.nan

    def inside_bloch_ball(r):
        return np.linalg.norm(r) <= 1 + ball_tol

    def is_duplicate(r, fixed_points):
        for fp in fixed_points:
            if np.linalg.norm(r - fp["point"]) < dedup_tol:
                return True
        return False

    # ------------------------------------------------------------
    # Build initial seeds
    # ------------------------------------------------------------

    seeds = []

    grid = np.linspace(-1, 1, n_grid)

    for x0 in grid:
        for y0 in grid:
            for z0 in grid:
                r0 = np.array([x0, y0, z0], dtype=float)
                if np.linalg.norm(r0) <= 1 + 1e-12:
                    seeds.append(r0)

    rng = np.random.default_rng(seed)

    # Random points uniformly inside the Bloch ball
    for _ in range(n_random):
        direction = rng.normal(size=3)
        direction /= np.linalg.norm(direction)

        radius = rng.random() ** (1 / 3)
        r0 = radius * direction

        seeds.append(r0)

    # Also explicitly include common special points
    special_points = [
        [0, 0, 0],
        [1, 0, 0],
        [-1, 0, 0],
        [0, 1, 0],
        [0, -1, 0],
        [0, 0, 1],
        [0, 0, -1],
        [1 / np.sqrt(3), 1 / np.sqrt(3), 1 / np.sqrt(3)],
        [-1 / np.sqrt(3), 1 / np.sqrt(3), 1 / np.sqrt(3)],
        [1 / np.sqrt(3), -1 / np.sqrt(3), 1 / np.sqrt(3)],
        [1 / np.sqrt(3), 1 / np.sqrt(3), -1 / np.sqrt(3)],
    ]

    for p in special_points:
        seeds.append(np.array(p, dtype=float))

    # ------------------------------------------------------------
    # Solve fixed-point equations from many initial seeds
    # ------------------------------------------------------------

    fixed_points = []

    for r0 in seeds:
        try:
            result = root(F_np, r0, method="hybr")

            if not result.success:
                continue

            r = np.asarray(result.x, dtype=float)

            if not np.all(np.isfinite(r)):
                continue

            residual = np.linalg.norm(F_np(r))

            if residual > residual_tol:
                continue

            if not inside_bloch_ball(r):
                continue

            p_succ = psuc_np(r)

            if not np.isfinite(p_succ):
                continue

            if abs(p_succ) < psuc_tol:
                continue

            if is_duplicate(r, fixed_points):
                continue

            J = J_np(r)
            eigvals = np.linalg.eigvals(J)
            spectral_radius = np.max(np.abs(eigvals))

            bloch_norm = np.linalg.norm(r)
            l1_norm = np.sum(np.abs(r))

            is_magic = (l1_norm > 1 + 1e-8) and (bloch_norm <= 1 + ball_tol)
            is_stabilizer_region = l1_norm <= 1 + 1e-8
            is_stable = spectral_radius < 1 - stability_tol

            fixed_points.append({
                "point": r,
                "residual": residual,
                "psuc": p_succ,
                "bloch_norm": bloch_norm,
                "l1_norm": l1_norm,
                "is_magic": is_magic,
                "is_stabilizer_region": is_stabilizer_region,
                "jacobian": J,
                "jacobian_eigvals": eigvals,
                "spectral_radius": spectral_radius,
                "is_stable": is_stable,
            })

        except Exception:
            pass

    fixed_points = sorted(
        fixed_points,
        key=lambda fp: (
            not fp["is_stable"],
            not fp["is_magic"],
            fp["bloch_norm"],
            fp["l1_norm"],
        )
    )
    
    stable_magic_points = [
        fp for fp in fixed_points
        if fp["is_stable"] and fp["is_magic"]
    ]

    if len(stable_magic_points) == 0:
        print("No stable magic fixed points found.")
        print()
    else:
        print("Stable magic fixed points found:")
        for fp in stable_magic_points:
            print(fp["point"])
            print("spectral radius:", fp["spectral_radius"])
            print("|x|+|y|+|z|:", fp["l1_norm"])
            print()

# Stabilizer Generator: XX

In [4]:
logical_operator_list_xx = [
    # X_L from {IX, XI}, Z_L from {ZZ, YY}
    {'X': 'IX', 'Z': 'ZZ'},
    {'X': 'IX', 'Z': 'YY'},
    {'X': 'XI', 'Z': 'ZZ'},
    {'X': 'XI', 'Z': 'YY'},

    # X_L from {IX, XI}, Z_L from {ZY, YZ}
    {'X': 'IX', 'Z': 'ZY'},
    {'X': 'IX', 'Z': 'YZ'},
    {'X': 'XI', 'Z': 'ZY'},
    {'X': 'XI', 'Z': 'YZ'},

    # X_L from {ZZ, YY}, Z_L from {IX, XI}
    {'X': 'ZZ', 'Z': 'IX'},
    {'X': 'ZZ', 'Z': 'XI'},
    {'X': 'YY', 'Z': 'IX'},
    {'X': 'YY', 'Z': 'XI'},

    # X_L from {ZZ, YY}, Z_L from {ZY, YZ}
    {'X': 'ZZ', 'Z': 'ZY'},
    {'X': 'ZZ', 'Z': 'YZ'},
    {'X': 'YY', 'Z': 'ZY'},
    {'X': 'YY', 'Z': 'YZ'},

    # X_L from {ZY, YZ}, Z_L from {IX, XI}
    {'X': 'ZY', 'Z': 'IX'},
    {'X': 'ZY', 'Z': 'XI'},
    {'X': 'YZ', 'Z': 'IX'},
    {'X': 'YZ', 'Z': 'XI'},

    # X_L from {ZY, YZ}, Z_L from {ZZ, YY}
    {'X': 'ZY', 'Z': 'ZZ'},
    {'X': 'ZY', 'Z': 'YY'},
    {'X': 'YZ', 'Z': 'ZZ'},
    {'X': 'YZ', 'Z': 'YY'},
]

In [47]:
for logical_operator in logical_operator_list_xx:
    fixed_points = find_magic_fixed_points(
        generator_set=['XX'],
        logical_operator=logical_operator,
        normalize=False,   # set True only if Tx, Ty, Tz are unnormalized numerators
        n_grid=11,
        n_random=500,
    )

Logical Operator: {'X': 'IX', 'Z': 'ZZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IX', 'Z': 'YY'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'ZZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'YY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IX', 'Z': 'ZY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IX', 'Z': 'YZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'ZY'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'YZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZZ', 'Z': 'IX'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZZ', 'Z': 'XI'}
No stable magic fixed points found.

Logical Operator: {'X': 'YY', 'Z': 'IX'}
No stable magic fixed points found.

Logical Operator: {'X': 'YY', 'Z': 'XI'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZZ', 'Z': 'ZY'}
No stable magic fixed p

# Stabilizer Generator: XY

In [5]:
logical_operator_list_xy = [
    # X_L from {IY, XI}, Z_L from {YX, ZZ}
    {'X': 'IY', 'Z': 'YX'},
    {'X': 'IY', 'Z': 'ZZ'},
    {'X': 'XI', 'Z': 'YX'},
    {'X': 'XI', 'Z': 'ZZ'},

    # X_L from {IY, XI}, Z_L from {YZ, ZX}
    {'X': 'IY', 'Z': 'YZ'},
    {'X': 'IY', 'Z': 'ZX'},
    {'X': 'XI', 'Z': 'YZ'},
    {'X': 'XI', 'Z': 'ZX'},

    # X_L from {YX, ZZ}, Z_L from {IY, XI}
    {'X': 'YX', 'Z': 'IY'},
    {'X': 'YX', 'Z': 'XI'},
    {'X': 'ZZ', 'Z': 'IY'},
    {'X': 'ZZ', 'Z': 'XI'},

    # X_L from {YX, ZZ}, Z_L from {YZ, ZX}
    {'X': 'YX', 'Z': 'YZ'},
    {'X': 'YX', 'Z': 'ZX'},
    {'X': 'ZZ', 'Z': 'YZ'},
    {'X': 'ZZ', 'Z': 'ZX'},

    # X_L from {YZ, ZX}, Z_L from {IY, XI}
    {'X': 'YZ', 'Z': 'IY'},
    {'X': 'YZ', 'Z': 'XI'},
    {'X': 'ZX', 'Z': 'IY'},
    {'X': 'ZX', 'Z': 'XI'},

    # X_L from {YZ, ZX}, Z_L from {YX, ZZ}
    {'X': 'YZ', 'Z': 'YX'},
    {'X': 'YZ', 'Z': 'ZZ'},
    {'X': 'ZX', 'Z': 'YX'},
    {'X': 'ZX', 'Z': 'ZZ'},
]

In [48]:
for logical_operator in logical_operator_list_xy:
    fixed_points = find_magic_fixed_points(
        generator_set=['XY'],
        logical_operator=logical_operator,
        normalize=False,   # set True only if Tx, Ty, Tz are unnormalized numerators
        n_grid=11,
        n_random=500,
    )

Logical Operator: {'X': 'IY', 'Z': 'YX'}
No stable magic fixed points found.

Logical Operator: {'X': 'IY', 'Z': 'ZZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'YX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'ZZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IY', 'Z': 'YZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IY', 'Z': 'ZX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'YZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'ZX'}
No stable magic fixed points found.

Logical Operator: {'X': 'YX', 'Z': 'IY'}
No stable magic fixed points found.

Logical Operator: {'X': 'YX', 'Z': 'XI'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZZ', 'Z': 'IY'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZZ', 'Z': 'XI'}
No stable magic fixed points found.

Logical Operator: {'X': 'YX', 'Z': 'YZ'}
No stable magic fixed p

# Stabilizer Generator: XZ

In [45]:
logical_operator_list_xz = [
    # X_L from {IZ, XI}, Z_L from {YX, ZY}
    {'X': 'IZ', 'Z': 'YX'},
    {'X': 'IZ', 'Z': 'ZY'},
    {'X': 'XI', 'Z': 'YX'},
    {'X': 'XI', 'Z': 'ZY'},

    # X_L from {IZ, XI}, Z_L from {YY, ZX}
    {'X': 'IZ', 'Z': 'YY'},
    {'X': 'IZ', 'Z': 'ZX'},
    {'X': 'XI', 'Z': 'YY'},
    {'X': 'XI', 'Z': 'ZX'},

    # X_L from {YX, ZY}, Z_L from {IZ, XI}
    {'X': 'YX', 'Z': 'IZ'},
    {'X': 'YX', 'Z': 'XI'},
    {'X': 'ZY', 'Z': 'IZ'},
    {'X': 'ZY', 'Z': 'XI'},

    # X_L from {YX, ZY}, Z_L from {YY, ZX}
    {'X': 'YX', 'Z': 'YY'},
    {'X': 'YX', 'Z': 'ZX'},
    {'X': 'ZY', 'Z': 'YY'},
    {'X': 'ZY', 'Z': 'ZX'},

    # X_L from {YY, ZX}, Z_L from {IZ, XI}
    {'X': 'YY', 'Z': 'IZ'},
    {'X': 'YY', 'Z': 'XI'},
    {'X': 'ZX', 'Z': 'IZ'},
    {'X': 'ZX', 'Z': 'XI'},

    # X_L from {YY, ZX}, Z_L from {YX, ZY}
    {'X': 'YY', 'Z': 'YX'},
    {'X': 'YY', 'Z': 'ZY'},
    {'X': 'ZX', 'Z': 'YX'},
    {'X': 'ZX', 'Z': 'ZY'},
]

In [46]:
for logical_operator in logical_operator_list_xz:
    fixed_points = find_magic_fixed_points(
        generator_set=['XZ'],
        logical_operator=logical_operator,
        normalize=False,   # set True only if Tx, Ty, Tz are unnormalized numerators
        n_grid=11,
        n_random=500,
    )

Logical Operator: {'X': 'IZ', 'Z': 'YX'}
No stable magic fixed points found.

Logical Operator: {'X': 'IZ', 'Z': 'ZY'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'YX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'ZY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IZ', 'Z': 'YY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IZ', 'Z': 'ZX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'YY'}
No stable magic fixed points found.

Logical Operator: {'X': 'XI', 'Z': 'ZX'}
No stable magic fixed points found.

Logical Operator: {'X': 'YX', 'Z': 'IZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'YX', 'Z': 'XI'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZY', 'Z': 'IZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZY', 'Z': 'XI'}
No stable magic fixed points found.

Logical Operator: {'X': 'YX', 'Z': 'YY'}
No stable magic fixed p

# Stabilizer Generator: YX

In [7]:
logical_operator_list_yx = [
    # X_L from {IX, YI}, Z_L from {XY, ZZ}
    {'X': 'IX', 'Z': 'XY'},
    {'X': 'IX', 'Z': 'ZZ'},
    {'X': 'YI', 'Z': 'XY'},
    {'X': 'YI', 'Z': 'ZZ'},

    # X_L from {IX, YI}, Z_L from {XZ, ZY}
    {'X': 'IX', 'Z': 'XZ'},
    {'X': 'IX', 'Z': 'ZY'},
    {'X': 'YI', 'Z': 'XZ'},
    {'X': 'YI', 'Z': 'ZY'},

    # X_L from {XY, ZZ}, Z_L from {IX, YI}
    {'X': 'XY', 'Z': 'IX'},
    {'X': 'XY', 'Z': 'YI'},
    {'X': 'ZZ', 'Z': 'IX'},
    {'X': 'ZZ', 'Z': 'YI'},

    # X_L from {XY, ZZ}, Z_L from {XZ, ZY}
    {'X': 'XY', 'Z': 'XZ'},
    {'X': 'XY', 'Z': 'ZY'},
    {'X': 'ZZ', 'Z': 'XZ'},
    {'X': 'ZZ', 'Z': 'ZY'},

    # X_L from {XZ, ZY}, Z_L from {IX, YI}
    {'X': 'XZ', 'Z': 'IX'},
    {'X': 'XZ', 'Z': 'YI'},
    {'X': 'ZY', 'Z': 'IX'},
    {'X': 'ZY', 'Z': 'YI'},

    # X_L from {XZ, ZY}, Z_L from {XY, ZZ}
    {'X': 'XZ', 'Z': 'XY'},
    {'X': 'XZ', 'Z': 'ZZ'},
    {'X': 'ZY', 'Z': 'XY'},
    {'X': 'ZY', 'Z': 'ZZ'},
]

In [49]:
for logical_operator in logical_operator_list_yx:
    fixed_points = find_magic_fixed_points(
        generator_set=['YX'],
        logical_operator=logical_operator,
        normalize=False,   # set True only if Tx, Ty, Tz are unnormalized numerators
        n_grid=11,
        n_random=500,
    )

Logical Operator: {'X': 'IX', 'Z': 'XY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IX', 'Z': 'ZZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'XY'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'ZZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IX', 'Z': 'XZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IX', 'Z': 'ZY'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'XZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'ZY'}
No stable magic fixed points found.

Logical Operator: {'X': 'XY', 'Z': 'IX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XY', 'Z': 'YI'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZZ', 'Z': 'IX'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZZ', 'Z': 'YI'}
No stable magic fixed points found.

Logical Operator: {'X': 'XY', 'Z': 'XZ'}
No stable magic fixed p

# Stabilizer Generator: YY

In [8]:
logical_operator_list_yy = [
    # X_L from {IY, YI}, Z_L from {XX, ZZ}
    {'X': 'IY', 'Z': 'XX'},
    {'X': 'IY', 'Z': 'ZZ'},
    {'X': 'YI', 'Z': 'XX'},
    {'X': 'YI', 'Z': 'ZZ'},

    # X_L from {IY, YI}, Z_L from {XZ, ZX}
    {'X': 'IY', 'Z': 'XZ'},
    {'X': 'IY', 'Z': 'ZX'},
    {'X': 'YI', 'Z': 'XZ'},
    {'X': 'YI', 'Z': 'ZX'},

    # X_L from {XX, ZZ}, Z_L from {IY, YI}
    {'X': 'XX', 'Z': 'IY'},
    {'X': 'XX', 'Z': 'YI'},
    {'X': 'ZZ', 'Z': 'IY'},
    {'X': 'ZZ', 'Z': 'YI'},

    # X_L from {XX, ZZ}, Z_L from {XZ, ZX}
    {'X': 'XX', 'Z': 'XZ'},
    {'X': 'XX', 'Z': 'ZX'},
    {'X': 'ZZ', 'Z': 'XZ'},
    {'X': 'ZZ', 'Z': 'ZX'},

    # X_L from {XZ, ZX}, Z_L from {IY, YI}
    {'X': 'XZ', 'Z': 'IY'},
    {'X': 'XZ', 'Z': 'YI'},
    {'X': 'ZX', 'Z': 'IY'},
    {'X': 'ZX', 'Z': 'YI'},

    # X_L from {XZ, ZX}, Z_L from {XX, ZZ}
    {'X': 'XZ', 'Z': 'XX'},
    {'X': 'XZ', 'Z': 'ZZ'},
    {'X': 'ZX', 'Z': 'XX'},
    {'X': 'ZX', 'Z': 'ZZ'},
]

In [50]:
for logical_operator in logical_operator_list_yy:
    fixed_points = find_magic_fixed_points(
        generator_set=['YY'],
        logical_operator=logical_operator,
        normalize=False,   # set True only if Tx, Ty, Tz are unnormalized numerators
        n_grid=11,
        n_random=500,
    )

Logical Operator: {'X': 'IY', 'Z': 'XX'}
No stable magic fixed points found.

Logical Operator: {'X': 'IY', 'Z': 'ZZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'XX'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'ZZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IY', 'Z': 'XZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IY', 'Z': 'ZX'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'XZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'ZX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'IY'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'YI'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZZ', 'Z': 'IY'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZZ', 'Z': 'YI'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'XZ'}
No stable magic fixed p

# Stabilizer Generator: YZ

In [9]:
logical_operator_list_yz = [
    # X_L from {IZ, YI}, Z_L from {XX, ZY}
    {'X': 'IZ', 'Z': 'XX'},
    {'X': 'IZ', 'Z': 'ZY'},
    {'X': 'YI', 'Z': 'XX'},
    {'X': 'YI', 'Z': 'ZY'},

    # X_L from {IZ, YI}, Z_L from {XY, ZX}
    {'X': 'IZ', 'Z': 'XY'},
    {'X': 'IZ', 'Z': 'ZX'},
    {'X': 'YI', 'Z': 'XY'},
    {'X': 'YI', 'Z': 'ZX'},

    # X_L from {XX, ZY}, Z_L from {IZ, YI}
    {'X': 'XX', 'Z': 'IZ'},
    {'X': 'XX', 'Z': 'YI'},
    {'X': 'ZY', 'Z': 'IZ'},
    {'X': 'ZY', 'Z': 'YI'},

    # X_L from {XX, ZY}, Z_L from {XY, ZX}
    {'X': 'XX', 'Z': 'XY'},
    {'X': 'XX', 'Z': 'ZX'},
    {'X': 'ZY', 'Z': 'XY'},
    {'X': 'ZY', 'Z': 'ZX'},

    # X_L from {XY, ZX}, Z_L from {IZ, YI}
    {'X': 'XY', 'Z': 'IZ'},
    {'X': 'XY', 'Z': 'YI'},
    {'X': 'ZX', 'Z': 'IZ'},
    {'X': 'ZX', 'Z': 'YI'},

    # X_L from {XY, ZX}, Z_L from {XX, ZY}
    {'X': 'XY', 'Z': 'XX'},
    {'X': 'XY', 'Z': 'ZY'},
    {'X': 'ZX', 'Z': 'XX'},
    {'X': 'ZX', 'Z': 'ZY'},
]

In [51]:
for logical_operator in logical_operator_list_yz:
    fixed_points = find_magic_fixed_points(
        generator_set=['YZ'],
        logical_operator=logical_operator,
        normalize=False,   # set True only if Tx, Ty, Tz are unnormalized numerators
        n_grid=11,
        n_random=500,
    )

Logical Operator: {'X': 'IZ', 'Z': 'XX'}
No stable magic fixed points found.

Logical Operator: {'X': 'IZ', 'Z': 'ZY'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'XX'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'ZY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IZ', 'Z': 'XY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IZ', 'Z': 'ZX'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'XY'}
No stable magic fixed points found.

Logical Operator: {'X': 'YI', 'Z': 'ZX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'IZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'YI'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZY', 'Z': 'IZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZY', 'Z': 'YI'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'XY'}
No stable magic fixed p

# Stabilizer Generator: ZX

In [10]:
logical_operator_list_zx = [
    # X_L from {IX, ZI}, Z_L from {XY, YZ}
    {'X': 'IX', 'Z': 'XY'},
    {'X': 'IX', 'Z': 'YZ'},
    {'X': 'ZI', 'Z': 'XY'},
    {'X': 'ZI', 'Z': 'YZ'},

    # X_L from {IX, ZI}, Z_L from {XZ, YY}
    {'X': 'IX', 'Z': 'XZ'},
    {'X': 'IX', 'Z': 'YY'},
    {'X': 'ZI', 'Z': 'XZ'},
    {'X': 'ZI', 'Z': 'YY'},

    # X_L from {XY, YZ}, Z_L from {IX, ZI}
    {'X': 'XY', 'Z': 'IX'},
    {'X': 'XY', 'Z': 'ZI'},
    {'X': 'YZ', 'Z': 'IX'},
    {'X': 'YZ', 'Z': 'ZI'},

    # X_L from {XY, YZ}, Z_L from {XZ, YY}
    {'X': 'XY', 'Z': 'XZ'},
    {'X': 'XY', 'Z': 'YY'},
    {'X': 'YZ', 'Z': 'XZ'},
    {'X': 'YZ', 'Z': 'YY'},

    # X_L from {XZ, YY}, Z_L from {IX, ZI}
    {'X': 'XZ', 'Z': 'IX'},
    {'X': 'XZ', 'Z': 'ZI'},
    {'X': 'YY', 'Z': 'IX'},
    {'X': 'YY', 'Z': 'ZI'},

    # X_L from {XZ, YY}, Z_L from {XY, YZ}
    {'X': 'XZ', 'Z': 'XY'},
    {'X': 'XZ', 'Z': 'YZ'},
    {'X': 'YY', 'Z': 'XY'},
    {'X': 'YY', 'Z': 'YZ'},
]

In [52]:
for logical_operator in logical_operator_list_zx:
    fixed_points = find_magic_fixed_points(
        generator_set=['ZX'],
        logical_operator=logical_operator,
        normalize=False,   # set True only if Tx, Ty, Tz are unnormalized numerators
        n_grid=11,
        n_random=500,
    )

Logical Operator: {'X': 'IX', 'Z': 'XY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IX', 'Z': 'YZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'XY'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'YZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IX', 'Z': 'XZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IX', 'Z': 'YY'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'XZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'YY'}
No stable magic fixed points found.

Logical Operator: {'X': 'XY', 'Z': 'IX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XY', 'Z': 'ZI'}
No stable magic fixed points found.

Logical Operator: {'X': 'YZ', 'Z': 'IX'}
No stable magic fixed points found.

Logical Operator: {'X': 'YZ', 'Z': 'ZI'}
No stable magic fixed points found.

Logical Operator: {'X': 'XY', 'Z': 'XZ'}
No stable magic fixed p

# Stabilizer Generator ZY

In [11]:
logical_operator_list_zy = [
    # X_L from {IY, ZI}, Z_L from {XX, YZ}
    {'X': 'IY', 'Z': 'XX'},
    {'X': 'IY', 'Z': 'YZ'},
    {'X': 'ZI', 'Z': 'XX'},
    {'X': 'ZI', 'Z': 'YZ'},

    # X_L from {IY, ZI}, Z_L from {XZ, YX}
    {'X': 'IY', 'Z': 'XZ'},
    {'X': 'IY', 'Z': 'YX'},
    {'X': 'ZI', 'Z': 'XZ'},
    {'X': 'ZI', 'Z': 'YX'},

    # X_L from {XX, YZ}, Z_L from {IY, ZI}
    {'X': 'XX', 'Z': 'IY'},
    {'X': 'XX', 'Z': 'ZI'},
    {'X': 'YZ', 'Z': 'IY'},
    {'X': 'YZ', 'Z': 'ZI'},

    # X_L from {XX, YZ}, Z_L from {XZ, YX}
    {'X': 'XX', 'Z': 'XZ'},
    {'X': 'XX', 'Z': 'YX'},
    {'X': 'YZ', 'Z': 'XZ'},
    {'X': 'YZ', 'Z': 'YX'},

    # X_L from {XZ, YX}, Z_L from {IY, ZI}
    {'X': 'XZ', 'Z': 'IY'},
    {'X': 'XZ', 'Z': 'ZI'},
    {'X': 'YX', 'Z': 'IY'},
    {'X': 'YX', 'Z': 'ZI'},

    # X_L from {XZ, YX}, Z_L from {XX, YZ}
    {'X': 'XZ', 'Z': 'XX'},
    {'X': 'XZ', 'Z': 'YZ'},
    {'X': 'YX', 'Z': 'XX'},
    {'X': 'YX', 'Z': 'YZ'},
]

In [53]:
for logical_operator in logical_operator_list_zy:
    fixed_points = find_magic_fixed_points(
        generator_set=['ZY'],
        logical_operator=logical_operator,
        normalize=False,   # set True only if Tx, Ty, Tz are unnormalized numerators
        n_grid=11,
        n_random=500,
    )

Logical Operator: {'X': 'IY', 'Z': 'XX'}
No stable magic fixed points found.

Logical Operator: {'X': 'IY', 'Z': 'YZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'XX'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'YZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IY', 'Z': 'XZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'IY', 'Z': 'YX'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'XZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'YX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'IY'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'ZI'}
No stable magic fixed points found.

Logical Operator: {'X': 'YZ', 'Z': 'IY'}
No stable magic fixed points found.

Logical Operator: {'X': 'YZ', 'Z': 'ZI'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'XZ'}
No stable magic fixed p

# Stabilizer Generator: ZZ

In [12]:
logical_operator_list_zz = [
    # X_L from {IZ, ZI}, Z_L from {XX, YY}
    {'X': 'IZ', 'Z': 'XX'},
    {'X': 'IZ', 'Z': 'YY'},
    {'X': 'ZI', 'Z': 'XX'},
    {'X': 'ZI', 'Z': 'YY'},

    # X_L from {IZ, ZI}, Z_L from {XY, YX}
    {'X': 'IZ', 'Z': 'XY'},
    {'X': 'IZ', 'Z': 'YX'},
    {'X': 'ZI', 'Z': 'XY'},
    {'X': 'ZI', 'Z': 'YX'},

    # X_L from {XX, YY}, Z_L from {IZ, ZI}
    {'X': 'XX', 'Z': 'IZ'},
    {'X': 'XX', 'Z': 'ZI'},
    {'X': 'YY', 'Z': 'IZ'},
    {'X': 'YY', 'Z': 'ZI'},

    # X_L from {XX, YY}, Z_L from {XY, YX}
    {'X': 'XX', 'Z': 'XY'},
    {'X': 'XX', 'Z': 'YX'},
    {'X': 'YY', 'Z': 'XY'},
    {'X': 'YY', 'Z': 'YX'},

    # X_L from {XY, YX}, Z_L from {IZ, ZI}
    {'X': 'XY', 'Z': 'IZ'},
    {'X': 'XY', 'Z': 'ZI'},
    {'X': 'YX', 'Z': 'IZ'},
    {'X': 'YX', 'Z': 'ZI'},

    # X_L from {XY, YX}, Z_L from {XX, YY}
    {'X': 'XY', 'Z': 'XX'},
    {'X': 'XY', 'Z': 'YY'},
    {'X': 'YX', 'Z': 'XX'},
    {'X': 'YX', 'Z': 'YY'},
]

In [54]:
for logical_operator in logical_operator_list_zz:
    fixed_points = find_magic_fixed_points(
        generator_set=['ZZ'],
        logical_operator=logical_operator,
        normalize=False,   # set True only if Tx, Ty, Tz are unnormalized numerators
        n_grid=11,
        n_random=500,
    )

Logical Operator: {'X': 'IZ', 'Z': 'XX'}
No stable magic fixed points found.

Logical Operator: {'X': 'IZ', 'Z': 'YY'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'XX'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'YY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IZ', 'Z': 'XY'}
No stable magic fixed points found.

Logical Operator: {'X': 'IZ', 'Z': 'YX'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'XY'}
No stable magic fixed points found.

Logical Operator: {'X': 'ZI', 'Z': 'YX'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'IZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'ZI'}
No stable magic fixed points found.

Logical Operator: {'X': 'YY', 'Z': 'IZ'}
No stable magic fixed points found.

Logical Operator: {'X': 'YY', 'Z': 'ZI'}
No stable magic fixed points found.

Logical Operator: {'X': 'XX', 'Z': 'XY'}
No stable magic fixed p